In [1]:
import sqlite3  # 读取sqlite数据库的
from pathlib import Path # 现代化处理路径的

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent  # notebook放在notebook文件里面上一级是根目录
DB_PATH = ROOT / "data" / "signal.db"  # 数据库文件
IMG_DIR = ROOT / "image"
IMG_DIR.mkdir(exist_ok = True)

assert DB_PATH.exists(), f"找不到数据库文件{DB_PATH}"

FONTS = ['TeX Gyre Termes', "Noto Serif CJK SC"]
plt.rcParams.update({
    "font.family": FONTS, "axes.unicode_minus": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y",
    "axes.axisbelow": True, "grid.color": "#E5E7EB",
    "axes.edgecolor": "#9CA3AF",
    "xtick.color": "#4B5563", "ytick.color": "#4B5563",
    "xtick.major.size": 0, "ytick.major.size": 0,
    "legend.frameon": False,
    "figure.dpi": 100, "savefig.dpi": 300,
})
# 配色：灰色打底，一种颜色强调一件事，红色只表示警示
MAIN, MAIN_LIGHT, GRAY, RED = "#1D6FA5", "#DCEAF5", "#C4CAD3", "#D64545"
INK, MUTED, FAINT = "#1F2937", "#6B7280", "#9CA3AF"
MAIN_DARK, MAIN_MID, GRAY_DARK, AMBER = "#174E73", "#7FB3D9", "#8E99A6", "#B7791F"
AREA_COLOR = {"西塔": MAIN, "太原街商圈": "#D9822B", "中街商圈": "#3B8C66",          # 同一区域在所有图里同色
              "沈阳北站": "#7A5EA8", "沈阳世博园": "#B04A6A", "棋盘山风景区": "#6B7F2A"}
SEG_COLOR = {"市内来访": MAIN, "周边居民": MAIN_MID, "外地来客": "#D9822B", "市外来访": "#D9822B",
             "过夜地不明": GRAY}                                                 # “不明”永远是灰色：不是一类人，是不知道
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
con = sqlite3.connect(DB_PATH)             # 整个 notebook 共用一个数据库连接

In [2]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name",con)
tables['行数']=[
    pd.read_sql(f"SELECT COUNT(*) AS n FROM {t}", con)["n"][0]
    for t in tables["name"]
]
raw = pd.read_sql("SELECT * FROM ods_signal", con)
raw.head()

,timestamp,imsi,lac_id,cell_id,phone,timestamp1,tmp0,tmp1,nid,npid
0,1538498482636,460000095007329090,16789.0,67567924.0,86137666647316,1538498481670,None,,4,#*#6137
1,1538502547368,460000095007329090,16789.0,67567924.0,86137666647316,1538502547366,None,,2,#*#6137
2,1538481509242,460000095007329090,16789.0,67567924.0,86137666647316,1538481509232,None,,20,#*#6137
3,1538526105074,460000095007329090,16789.0,67200820.0,86137666647316,1538526104550,None,,4,#*#6137
4,1538526153986,460000095007329090,16789.0,67200820.0,86137666647316,1538526153940,None,,20,#*#6137


In [3]:
assert raw.shape == (532_828, 10), f"行列数不对： {raw.shape},应该是：(532828, 10)"
print("数量核对准确： 532828行 x 10列")

数量核对准确： 532828行 x 10列


In [1]:
valid = raw.dropna(subset=['imsi'])
valid = valid[valid['imsi'].str.fullmatch(r"\d{18}")
              & (valid['phone'] != "6666")]
# 只看那个imsi 配过那个手机号，这组对应关系，同一对出现多少次不重要，所以要先去重
pairs = valid[['imsi','phone']].drop_duplicates()
# 裁剪出前15位，作为另外一种认证方式
pairs['imsi15']  = pairs['imsi'].str[:15]

per18 = pairs.groupby("imsi")['phone'].nunique()
per15 = pairs.groupby('imsi15')['phone'].nunique()

key_check = pd.DataFrame({
    "分组方式": ['按前15位',"按完整18位"],
    "ID个数": [per15.shape[0], per18.shape[0]],
    "对应多个手机号的ID": [(per15 > 1).sum(), (per18 > 1).sum()]
})
key_check['占比'] = (key_check['对应多个手机号的ID'] / key_check['ID个数']).round(3)
key_check.to_sql("ads_key_check", con, if_exists="replace",index=False)

NameError: name 'raw' is not defined

In [ ]:
multi = per15[per15 > 1].index
affected = pairs.loc[pairs['imsi15'].isin(multi),'imsi'].nunique()
suffuix_kinds = pairs['imsi'].str[-3:].nunique()
print(f"对应多个手机号的前15位分组："
    f"{len(multi)}组，里面装着{affected:,}个不同的18位IMSI,"
    f"占全部用户的{affected / len(per18):.1%}")
print(f"后三位一共有{suffuix_kinds}种取值")

In [ ]:
k = pd.read_sql("SELECT * FROM ads_key_check", con)
raws = [(f"按前 15 位分组\n（{k.loc[0, 'ID个数']:,} 组）",
         k.loc[0, "对应多个手机号的ID"],
         k.loc[0, "ID个数"], "组对应多个手机号"),
        (f"被牵连的用户\n（{len(per18):,} 个 18 位 IMSI）", affected, len(per18),
         "个用户会被错误合并"),
        (f"按完整 18 位\n（{k.loc[1, 'ID个数']:,} 个）", k.loc[1, "对应多个手机号的ID"],
         k.loc[1, "ID个数"], "个对应多个手机号")]
rows